# 05 - label-noise ceiling

How high can R² go on these labels at all? Each $T_S$ is the mean of 3 replicates, so even a
perfect model is scored against noisy labels. The noise ceiling is the R² that perfect model
would get:

$$R^2_{\mathrm{ceiling}} = 1 - \frac{\bar v}{\mathrm{Var}(y)}$$

with $y = \ln T_S$ (the probe's target), $\bar v$ the mean label-noise variance over the test
fold, and Var with ddof = 0, the same denominator `r2_score` uses. Folds are the probe's own
GroupKFold split from `results/folds.json`, not a new one.

Per-construct noise $v_i$, two ways:
- **(A) delta method:** $v_i = (SE_i / m_i)^2 / c^2$ with $SE_i = s_i / \sqrt{3}$ and $c = 1$
  for the natural log.
- **(B) lognormal simulation:** each replicate ~ LogNormal with mean $m_i$ and SD $s_i$;
  label = ln(mean of 3 draws); $v_i$ = variance over 10,000 draws.

Choices:
- 8 constructs have no SD. They stay in Var(y) (the probe scored them) and get their fold's
  average noise.
- Step 3 (probe R² / ceiling) is skipped for now.
- Step 4 is a diagnostic only: the ceiling without the CV > 0.5 constructs. The training set
  does not change.

In [1]:
import json, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

warnings.simplefilter("ignore", UserWarning)        # openpyxl "unknown extension" noise
REPO = Path.cwd().parent if Path.cwd().name == "probing" else Path.cwd()
sys.path.insert(0, str(REPO / "probing"))
from plot_strength_ranked import read_excel_unlocked  # survives the xlsx being open in Excel

SEED = 42
N_REP = 3          # Chen: each Ts is the mean of 3 replicates
N_SIM = 10_000     # lognormal draws per construct (Step 1B)
N_MC = 2_000       # Monte Carlo draws per fold (Step 2)
CV_FLAG = 0.5      # s/m above this: the delta method (A) is unreliable

CSV = REPO / "probing/data/chen2013_window_up90.csv"
XLSX = [REPO / "Chen_2013_terminators/41592_2013_BFnmeth2515_MOESM206_ESM.xlsx",
        REPO / "Chen_2013_terminators/41592_2013_BFnmeth2515_MOESM207_ESM.xlsx"]
FOLDS = REPO / "probing/results/folds.json"
OUT = REPO / "probing/results/noise_ceiling.csv"

## Step 0 - check the inputs

In [2]:
df = pd.read_csv(CSV)
xl = pd.concat([read_excel_unlocked(p) for p in XLSX], ignore_index=True)
df = df.merge(xl[["Name", "Average Strength", "Standard Deviation of Strength"]],
              left_on="name", right_on="Name", how="left", validate="1:1")
assert df["Name"].notna().all(), "a construct in the CSV is missing from the xlsx"
assert np.allclose(df["strength"], df["Average Strength"]), "CSV Ts and xlsx Ts disagree"

m = df["strength"].to_numpy(float)                       # mean Ts of the 3 replicates
s = pd.to_numeric(df["Standard Deviation of Strength"], errors="coerce").to_numpy(float)
y = np.log(m)                                            # the probe's target: natural log
LOG_C = 1.0                                              # c = 1 for ln (it would be ln 10 for log10)
has_sd = np.isfinite(s)

folds = json.loads(FOLDS.read_text())["schemes"]["GroupKFold"]
tests = [np.array(te) for _, te in folds]
assert sorted(np.concatenate(tests).tolist()) == list(range(len(df))), "folds must cover every row once"

print("labels  :", CSV.relative_to(REPO), " -> Ts column 'strength'")
print("SD      :", " + ".join(p.name for p in XLSX))
print("          -> column 'Standard Deviation of Strength' (raw Ts scale), joined on name")
print("folds   :", FOLDS.relative_to(REPO), " -> 'GroupKFold',", len(tests), "folds, grouped by hairpin")
print("rows    :", len(df), "| valid SD:", int(has_sd.sum()), "| SD = 0:", int((s == 0).sum()))
print("no SD   :", ", ".join(df.loc[~has_sd, "name"]))
print("log base: e  (y = np.log(Ts), as in probe_rnafm.ipynb)")

labels  : probing\data\chen2013_window_up90.csv  -> Ts column 'strength'
SD      : 41592_2013_BFnmeth2515_MOESM206_ESM.xlsx + 41592_2013_BFnmeth2515_MOESM207_ESM.xlsx
          -> column 'Standard Deviation of Strength' (raw Ts scale), joined on name
folds   : probing\results\folds.json  -> 'GroupKFold', 10 folds, grouped by hairpin
rows    : 582 | valid SD: 574 | SD = 0: 2
no SD   : L3S2P43, L3S2P33, L3S3P32, L1U1H03, L1U2H01, L1U5H04, L1U2H09, L1U1H05
log base: e  (y = np.log(Ts), as in probe_rnafm.ipynb)


## Step 1 - noise variance of each label

The label is a mean of 3 replicates, so its noise is $SE_i = s_i / \sqrt{3}$, not $s_i$.

In [3]:
se = s / np.sqrt(N_REP)
v_A = (se / m) ** 2 / LOG_C ** 2                         # (A) delta method

# (B) each replicate ~ LogNormal with mean m and SD s; label = log(arithmetic mean of 3 draws)
rng = np.random.default_rng(SEED)
sigma2 = np.log1p((s / m) ** 2)
mu = np.log(m) - sigma2 / 2
v_B = np.full(len(df), np.nan)
for i in np.flatnonzero(has_sd):
    reps = np.exp(mu[i] + np.sqrt(sigma2[i]) * rng.standard_normal((N_SIM, N_REP)))
    v_B[i] = np.var(np.log(reps.mean(axis=1)) / LOG_C)

cv = s / m
flag = cv > CV_FLAG                                      # NaN compares False: no-SD rows are not flagged

print("per-construct label-noise variance, ln scale")
for tag, v in (("A", v_A), ("B", v_B)):
    q = np.nanpercentile(v, [50, 90, 99])
    print(f"  ({tag}) median {q[0]:.4f}   p90 {q[1]:.4f}   p99 {q[2]:.4f}   mean {np.nanmean(v):.4f}")
pos = has_sd & (s > 0)
print(f"  median v_B / v_A: {np.median(v_B[pos] / v_A[pos]):.3f}")
print(f"flagged CV > {CV_FLAG}: {int(flag.sum())} constructs, (A) unreliable for these")
print(f"  mean v on flagged rows: (A) {v_A[flag].mean():.4f}   (B) {v_B[flag].mean():.4f}")
print("  " + ", ".join(df.loc[flag, "name"]))

per-construct label-noise variance, ln scale
  (A) median 0.0152   p90 0.0710   p99 0.1324   mean 0.0281
  (B) median 0.0148   p90 0.0685   p99 0.1218   mean 0.0270
  median v_B / v_A: 0.986
flagged CV > 0.5: 30 constructs, (A) unreliable for these
  mean v on flagged rows: (A) 0.1184   (B) 0.1085
  ECK120010818, ECK120010869, BBa_K088008, ECK120011170, ECK120051408, ECK120030798, frd_ampC, ECK120010784, ECK120048901, ECK120010821, ECK120010802, ECK125122040, ECK120010829, trp_t, orfL, ECK120010836-R, ECK120026315-R, ECK120010782-R, ECK120010844-R, ECK120011169-R, ECK120020525-R, L3S2P22, L3S2P52, L3S3P46, L2U7H06, L2U6H11, L1U7H08, L2U2H01, L2U1H01, L2U8H06


## Step 2 - ceiling per test fold

`ceiling()` gives the analytic ceiling plus a Monte Carlo check: add $\epsilon_i \sim N(0, v_i)$
to the true labels and score with `r2_score`, 2,000 times. The MC mean has to match the analytic
ceiling; if it doesn't, there is a bug.

In [4]:
def ceiling(idx, v, rng):
    """Noise ceiling on rows idx. v = per-row noise variance (NaN = unknown SD)."""
    y_t, v_t = y[idx], v[idx]
    var_obs = np.var(y_t)                                # ddof=0, as in r2_score
    v_bar = np.nanmean(v_t)                              # rows without an SD get typical noise
    r2c = 1 - v_bar / var_obs
    v_mc = np.where(np.isfinite(v_t), v_t, v_bar)
    eps = rng.normal(0.0, np.sqrt(v_mc), size=(N_MC, len(idx)))
    # one r2_score call: every column is one MC draw
    mc = r2_score(np.tile(y_t, (N_MC, 1)).T, (y_t - eps).T, multioutput="raw_values")
    assert abs(mc.mean() - r2c) < 5 * mc.std() / np.sqrt(N_MC) + 1e-9, "MC mean != ceiling: bug"
    return dict(n_test=len(idx), var_obs=var_obs, v_bar=v_bar, R2_ceiling=r2c,
                MC_mean=mc.mean(), MC_lo=np.percentile(mc, 2.5), MC_hi=np.percentile(mc, 97.5),
                r_ceiling=np.sqrt(r2c) if r2c > 0 else np.nan)

rng_mc = np.random.default_rng(SEED + 1)
splits = [(k, te) for k, te in enumerate(tests)] + [("pooled", np.arange(len(df)))]
rows = []
for fold, idx in splits:
    row = {"fold": fold}
    for tag, v in (("A", v_A), ("B", v_B)):
        r = ceiling(idx, v, rng_mc)
        row.update(n_test=r["n_test"], var_obs=r["var_obs"])
        row.update({f"v_bar_{tag}": r["v_bar"], f"R2_ceiling_{tag}": r["R2_ceiling"],
                    f"MC_mean_{tag}": r["MC_mean"], f"MC_p2.5_{tag}": r["MC_lo"],
                    f"MC_p97.5_{tag}": r["MC_hi"], f"r_ceiling_{tag}": r["r_ceiling"]})
    rows.append(row)

show = ["fold", "n_test", "var_obs", "v_bar_A", "v_bar_B", "R2_ceiling_A", "MC_mean_A",
        "R2_ceiling_B", "MC_mean_B", "r_ceiling_B"]
print(pd.DataFrame(rows)[show].to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("MC check passed: every MC mean matches its analytic ceiling")

  fold  n_test  var_obs  v_bar_A  v_bar_B  R2_ceiling_A  MC_mean_A  R2_ceiling_B  MC_mean_B  r_ceiling_B
     0      59    4.128    0.025    0.024         0.994      0.994         0.994      0.994        0.997
     1      59    3.765    0.023    0.022         0.994      0.994         0.994      0.994        0.997
     2      58    3.876    0.032    0.030         0.992      0.992         0.992      0.992        0.996
     3      58    2.576    0.028    0.027         0.989      0.989         0.989      0.990        0.995
     4      58    2.533    0.036    0.035         0.986      0.985         0.986      0.986        0.993
     5      58    1.757    0.033    0.032         0.981      0.981         0.982      0.982        0.991
     6      58    2.856    0.029    0.028         0.990      0.990         0.990      0.990        0.995
     7      58    2.453    0.022    0.021         0.991      0.991         0.991      0.991        0.996
     8      58    1.342    0.028    0.027         0.979

## Step 4 - diagnostic: without the flagged high-CV constructs

Same folds, flagged rows removed from both Var(y) and $\bar v$. Diagnostic only; nothing is
retrained.

In [5]:
rng_d = np.random.default_rng(SEED + 2)
for row, (fold, idx) in zip(rows, splits):
    sub = idx[~flag[idx]]
    row["n_test_noflag"] = len(sub)
    for tag, v in (("A", v_A), ("B", v_B)):
        row[f"R2_ceiling_{tag}_noflag"] = ceiling(sub, v, rng_d)["R2_ceiling"]

show = ["fold", "n_test", "n_test_noflag", "R2_ceiling_A", "R2_ceiling_A_noflag",
        "R2_ceiling_B", "R2_ceiling_B_noflag"]
print(pd.DataFrame(rows)[show].to_string(index=False, float_format=lambda x: f"{x:.3f}"))

  fold  n_test  n_test_noflag  R2_ceiling_A  R2_ceiling_A_noflag  R2_ceiling_B  R2_ceiling_B_noflag
     0      59             56         0.994                0.995         0.994                0.995
     1      59             56         0.994                0.996         0.994                0.996
     2      58             55         0.992                0.994         0.992                0.994
     3      58             53         0.989                0.991         0.989                0.991
     4      58             53         0.986                0.989         0.986                0.989
     5      58             54         0.981                0.985         0.982                0.986
     6      58             55         0.990                0.991         0.990                0.992
     7      58             58         0.991                0.991         0.991                0.991
     8      58             56         0.979                0.982         0.980                0.983


## Output

In [6]:
res = pd.DataFrame(rows)
res.to_csv(OUT, index=False)
print("saved", OUT.relative_to(REPO), res.shape)

per = res[res["fold"] != "pooled"]
pool = res[res["fold"] == "pooled"].iloc[0]
print()
print("=== label-noise ceiling | y = ln(Ts) | GroupKFold, 10 folds ===")
for tag, name in (("A", "delta method"), ("B", "lognormal sim")):
    c = per[f"R2_ceiling_{tag}"]
    print(f"({tag}) {name:<14} R2 ceiling {c.mean():.3f} +/- {c.std(ddof=0):.3f} across folds"
          f" | pooled {pool[f'R2_ceiling_{tag}']:.3f} | pooled r ceiling {pool[f'r_ceiling_{tag}']:.3f}")
print(f"flagged constructs (CV > {CV_FLAG}): {int(flag.sum())}")
for tag in "AB":
    a, b = per[f"R2_ceiling_{tag}"].mean(), per[f"R2_ceiling_{tag}_noflag"].mean()
    print(f"Step 4 ({tag}) without flagged: fold mean {b:.3f} (vs {a:.3f}),"
          f" pooled {pool[f'R2_ceiling_{tag}_noflag']:.3f} (vs {pool[f'R2_ceiling_{tag}']:.3f})")

saved probing\results\noise_ceiling.csv (11, 18)

=== label-noise ceiling | y = ln(Ts) | GroupKFold, 10 folds ===
(A) delta method   R2 ceiling 0.988 +/- 0.005 across folds | pooled 0.991 | pooled r ceiling 0.995
(B) lognormal sim  R2 ceiling 0.989 +/- 0.005 across folds | pooled 0.991 | pooled r ceiling 0.996
flagged constructs (CV > 0.5): 30
Step 4 (A) without flagged: fold mean 0.990 (vs 0.988), pooled 0.992 (vs 0.991)
Step 4 (B) without flagged: fold mean 0.991 (vs 0.989), pooled 0.993 (vs 0.991)


**Reading:** the label-noise ceiling is about **0.99** (fold mean 0.988 / 0.989 for A / B, pooled
0.991). The replicate noise is small next to the spread of $\ln T_S$: mean noise variance 0.028
against Var(y) = 3.0. So replicate noise is not what holds the probe back. A best GroupKFold
R² of 0.63 leaves most of the gap to the model and representation.

The two methods agree (median $v_B / v_A$ = 0.99). The 30 high-CV constructs barely move the
ceiling (+0.002).

Caveat: this only counts the noise the 3-replicate SD captures. Day-to-day or batch effects
that all 3 replicates share are not in it, so the real ceiling is lower than this. Even if the
SD column were already a standard error (3x the variance here), the ceiling would still be
about 0.97.

## Diagnostics - is 0.99 believable?

Checked because two things looked inconsistent with it: Chen's no-terminator control is reported
as $T_S$ = 1.00 ± 0.36, and the 95% CIs in the strength plot span more than a decade at the weak
end. Four checks, then the ceiling under each reading of the error column. Analytic only (no new
Monte Carlo); nothing above changes.

In [7]:
import ctypes, tempfile
from openpyxl import load_workbook
from scipy.stats import t as t_dist

def open_wb(path, data_only):
    """openpyxl version of read_excel_unlocked: copy with CopyFileW if Excel holds the lock."""
    try:
        return load_workbook(path, data_only=data_only)
    except PermissionError:
        tmp = Path(tempfile.gettempdir()) / ("_locked_" + path.name)
        if not ctypes.windll.kernel32.CopyFileW(str(path.resolve()), str(tmp), False):
            raise ctypes.WinError()
        return load_workbook(tmp, data_only=data_only)

T2 = t_dist.ppf(0.975, N_REP - 1)                        # 4.303, the t of a 95% CI from 3 replicates

print("Check 1 - is the error column an SD or an SE?")
for p in XLSX:
    wb = open_wb(p, data_only=False)
    ws = wb.worksheets[0]
    cols = [c for c in ws[1] if c.value and "Strength" in str(c.value)]
    n_comments = sum(1 for sh in wb.worksheets for row in sh.iter_rows() for c in row if c.comment)
    print(f"  {p.name}  sheets {wb.sheetnames}  | cell comments: {n_comments}")
    for c in cols:
        print(f"    col {c.column_letter}: {c.value!r}")

# the 95% CI is not Chen's: the 'Strength CI' sheet was added to this copy and is built from the SD column
ws_f = open_wb(XLSX[1], data_only=False)["Strength CI"]
print("  'Strength CI' sheet, formulas:")
for addr in ("B3", "B4", "B5"):
    print(f"    {ws_f['A' + addr[1:]].value:<17} {addr} = {ws_f[addr].value}")
for col in "EFGH":
    print(f"    {ws_f[col + '7'].value:<17} {col}8 = {ws_f[col + '8'].value}")

ci = read_excel_unlocked(XLSX[1], sheet_name="Strength CI", header=6).iloc[:, :8]
ci.columns = ["rank", "name", "Ts", "SD", "half", "lo", "hi", "lo_plot"]
ci = ci.dropna(subset=["name"]).astype({"Ts": float, "SD": float, "lo": float, "hi": float, "lo_plot": float})
pos_sd = ci["SD"] > 0
ratio = (ci["hi"] - ci["lo"])[pos_sd] / ci["SD"][pos_sd]
print(f"  CI width / column value: median {ratio.median():.3f} (min {ratio.min():.3f}, max {ratio.max():.3f}),"
      f" {int(pos_sd.sum())} synthetic rows")
print(f"    per-replicate SD with t: 2t/sqrt(3) = {2 * T2 / np.sqrt(3):.3f} | SE with t: 2t = {2 * T2:.3f}"
      f" | SE with normal: {2 * 1.96:.3f}")
print("    -> fixed by the sheet's own formula, so it cannot tell SD from SE")

ctrl = xl["Name"].astype(str).str.contains("control|no term|empty|none", case=False)
print(f"  no-terminator control row in the tables: {int(ctrl.sum())}")
near1 = np.argsort(np.abs(m - 1))[:3]
print("  closest to Ts = 1:", ", ".join(f"{df.name[i]} ({m[i]:.2f} +/- {s[i]:.2f})" for i in near1))

Check 1 - is the error column an SD or an SE?
  41592_2013_BFnmeth2515_MOESM206_ESM.xlsx  sheets ['Fig2a-natural']  | cell comments: 0
    col L: 'Average Strength'
    col R: 'Standard Deviation of Strength'


  41592_2013_BFnmeth2515_MOESM207_ESM.xlsx  sheets ['Sheet1', 'Strength CI']  | cell comments: 0
    col K: 'Average Strength'
    col Q: 'Standard Deviation of Strength'
  'Strength CI' sheet, formulas:
    Replicates (n)    B3 = 3
    Confidence level  B4 = 0.95
    t multiplier      B5 = =_xlfn.T.INV.2T(1-B4,B3-1)
    CI half-width     E8 = =$B$5*D8/SQRT($B$3)
    Lower 95%         F8 = =C8-E8
    Upper 95%         G8 = =C8+E8
    Lower (plot floor) H8 = =MAX(F8,C8/10)
  CI width / column value: median 4.968 (min 4.968, max 4.968), 255 synthetic rows
    per-replicate SD with t: 2t/sqrt(3) = 4.968 | SE with t: 2t = 8.605 | SE with normal: 3.920
    -> fixed by the sheet's own formula, so it cannot tell SD from SE
  no-terminator control row in the tables: 0
  closest to Ts = 1: L2U3H03 (1.00 +/- 0.44), L2U7H11 (0.99 +/- 0.29), ECK120010810 (0.99 +/- 0.08)


In [8]:
PCT = [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 100]
pct = lambda a: "  ".join(f"p{p}={v:.4f}" for p, v in zip(PCT, np.nanpercentile(a, PCT)))

v_sd = (s / m) ** 2 / N_REP                              # column read as SD of replicates (= v_A)
print(f"Check 2 - distribution, {int(has_sd.sum())} constructs with an SD")
print("  CV = s/m:")
print("   ", pct(cv))
print("  v_i on the ln scale (SD reading):")
print("   ", pct(v_sd))
print(f"  mean {np.nanmean(v_sd):.4f}   median {np.nanmedian(v_sd):.4f}   mean/median {np.nanmean(v_sd) / np.nanmedian(v_sd):.2f}")
vv = np.sort(v_sd[has_sd])[::-1]
k5, k10 = int(np.ceil(0.05 * len(vv))), int(np.ceil(0.10 * len(vv)))
print(f"  top 5% ({k5} constructs) hold {vv[:k5].sum() / vv.sum():.1%} of sum(v_i);"
      f" top 10% hold {vv[:k10].sum() / vv.sum():.1%}  (SE reading = 3x every v_i, same shares)")
top = (df.assign(Ts=m, err=s, CV=cv, v_i=v_sd).loc[has_sd]
         .nlargest(10, "v_i")[["name", "library", "Ts", "err", "CV", "v_i"]])
print("  10 largest v_i:")
print(top.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

Check 2 - distribution, 574 constructs with an SD
  CV = s/m:
    p0=0.0000  p1=0.0085  p5=0.0455  p10=0.0702  p25=0.1260  p50=0.2137  p75=0.3613  p90=0.4616  p95=0.5011  p99=0.6301  p100=0.9972
  v_i on the ln scale (SD reading):
    p0=0.0000  p1=0.0000  p5=0.0007  p10=0.0016  p25=0.0053  p50=0.0152  p75=0.0435  p90=0.0710  p95=0.0837  p99=0.1324  p100=0.3315
  mean 0.0281   median 0.0152   mean/median 1.85
  top 5% (29 constructs) hold 21.5% of sum(v_i); top 10% hold 35.4%  (SE reading = 3x every v_i, same shares)
  10 largest v_i:
          name   library      Ts     err     CV    v_i
       L3S3P46 synthetic 14.0656 14.0261 0.9972 0.3315
  ECK120011170   natural 15.5152 12.6871 0.8177 0.2229
       L1U7H08 synthetic  1.2186  0.8440 0.6926 0.1599
          orfL   natural  0.8387  0.5553 0.6621 0.1461
  ECK120010784   natural  5.5651  3.6542 0.6566 0.1437
       L2U6H11 synthetic  2.1251  1.3628 0.6413 0.1371
      frd_ampC   natural  8.6160  5.3930 0.6259 0.1306
  ECK120010821   na

In [9]:
print("Check 3 - implied SD from the CI bounds (synthetic only; this CI comes from the SD column)")
pos = ci["lo"] > 0
w = np.log(ci["hi"][pos]) - np.log(ci["lo"][pos])
sd_t, sd_z = w / (2 * T2), w / (2 * 1.96)
same = np.sqrt((ci["SD"][pos] / ci["Ts"][pos]) ** 2 / N_REP)
print(f"  rows with lower bound > 0: {int(pos.sum())} of {len(ci)}  ({int((~pos).sum())} drawn at the Ts/10 plot floor)")
print(f"  implied SD of the mean, ln scale, t = {T2:.3f}:  mean {sd_t.mean():.3f}  median {sd_t.median():.3f}")
print(f"  same rows, sqrt(v_i) from the SD column:     mean {same.mean():.3f}  median {same.median():.3f}")
print(f"  (converting with z = 1.96 instead of t:      mean {sd_z.mean():.3f}  median {sd_z.median():.3f})")
print(f"  sqrt(mean v_i) over all 574 = {np.sqrt(np.nanmean(v_sd)):.3f};  median sqrt(v_i) = {np.nanmedian(np.sqrt(v_sd)):.3f}")
span = np.log10(ci["hi"] / ci["lo_plot"])
print(f"  plotted CI span in decades: median {span.median():.2f}; >= 1 decade in {int((span >= 1).sum())} of {len(ci)} rows")

Check 3 - implied SD from the CI bounds (synthetic only; this CI comes from the SD column)
  rows with lower bound > 0: 217 of 265  (48 drawn at the Ts/10 plot floor)
  implied SD of the mean, ln scale, t = 4.303:  mean 0.135  median 0.096
  same rows, sqrt(v_i) from the SD column:     mean 0.099  median 0.091
  (converting with z = 1.96 instead of t:      mean 0.296  median 0.211)
  sqrt(mean v_i) over all 574 = 0.168;  median sqrt(v_i) = 0.123
  plotted CI span in decades: median 0.43; >= 1 decade in 76 of 265 rows


In [10]:
print("Check 4 - what Var(ln Ts) is made of")
print(f"  all 582, ddof=0: {np.var(y):.3f}   Ts min {m.min():.3f}  max {m.max():.2f}  (ln {y.min():.2f} .. {y.max():.2f})")
lo1, hi1 = np.percentile(m, [1, 99])
mid = (m >= lo1) & (m <= hi1)
print(f"  without the top and bottom 1% of Ts ({int((~mid).sum())} rows): {np.var(y[mid]):.3f}  (n = {int(mid.sum())})")
for lib in ("natural", "synthetic"):
    g = df["library"].to_numpy() == lib
    print(f"  {lib:<9} only: {np.var(y[g]):.3f}  (n = {int(g.sum())})")

Check 4 - what Var(ln Ts) is made of
  all 582, ddof=0: 3.014   Ts min 0.326  max 382.13  (ln -1.12 .. 5.95)
  without the top and bottom 1% of Ts (12 rows): 2.831  (n = 570)
  natural   only: 1.980  (n = 317)
  synthetic only: 4.250  (n = 265)


In [11]:
def analytic_ceiling(v):
    """Pooled and per-fold ceiling, no Monte Carlo. Rows without an SD get typical noise."""
    folds_c = [1 - np.nanmean(v[t]) / np.var(y[t]) for t in tests]
    return 1 - np.nanmean(v) / np.var(y), np.mean(folds_c), np.std(folds_c), min(folds_c)

readings = {
    "SD of replicates, delta      v=(s/m)^2/3":     (s / m) ** 2 / N_REP,
    "SD of replicates, lognormal  v=ln(1+(s/m)^2/3)": np.log1p((s / m) ** 2 / N_REP),
    "SE of the mean, delta        v=(s/m)^2":       (s / m) ** 2,
    "SE of the mean, lognormal    v=ln(1+(s/m)^2)": np.log1p((s / m) ** 2),
}
print("Ceiling under each reading of the error column")
print(f"  {'reading':<48}{'mean v':>8}{'pooled':>9}{'fold mean':>11}{'fold sd':>9}{'worst fold':>12}")
for name, v in readings.items():
    p_, fm, fs, fmin = analytic_ceiling(v)
    print(f"  {name:<48}{np.nanmean(v):>8.4f}{p_:>9.3f}{fm:>11.3f}{fs:>9.3f}{fmin:>12.3f}")

v_se = (s / m) ** 2
noisy = has_sd & (v_se > np.nanpercentile(v_se, 95))
rest = ~noisy
print("  bulk vs tail, SE reading (the pessimistic one):")
print(f"    all rows {1 - np.nanmean(v_se) / np.var(y):.3f} | without the noisiest 5% ({int(noisy.sum())} rows)"
      f" {1 - np.nanmean(v_se[rest]) / np.var(y[rest]):.3f} | median v instead of mean {1 - np.nanmedian(v_se) / np.var(y):.3f}")

Ceiling under each reading of the error column
  reading                                           mean v   pooled  fold mean  fold sd  worst fold
  SD of replicates, delta      v=(s/m)^2/3          0.0281    0.991      0.988    0.005       0.979
  SD of replicates, lognormal  v=ln(1+(s/m)^2/3)    0.0273    0.991      0.989    0.005       0.980
  SE of the mean, delta        v=(s/m)^2            0.0844    0.972      0.965    0.014       0.937
  SE of the mean, lognormal    v=ln(1+(s/m)^2)      0.0774    0.974      0.968    0.013       0.942
  bulk vs tail, SE reading (the pessimistic one):
    all rows 0.972 | without the noisiest 5% (29 rows) 0.977 | median v instead of mean 0.985


**Reading (diagnostics):** the arithmetic holds; the ceiling is high because $\ln T_S$ spreads
over three orders of magnitude (Var 3.0; 2.8 without the top and bottom 1%), not because the
noise is small.

- **SD or SE:** the header is `Standard Deviation of Strength`, with no units or footnote. The 95%
  CI is not Chen's: the `Strength CI` sheet was added to this copy and computes the CI from that
  column assuming it is a per-replicate SD (half-width = t·SD/√3, t = 4.30), so the CI cannot
  settle the question. Read at face value the /3 is right; Chen's Methods would confirm it.
- **The wide CIs** come from t = 4.30 with 2 df (2.2x a normal CI) and the $T_S$/10 floor for
  the 48 rows whose lower bound is ≤ 0. The median plotted CI spans 0.43 decades. If the
  control's ±0.36 is a per-replicate SD, its CV sits at about the 75th percentile of the
  construct CVs.
- **Ceiling:** 0.988 ± 0.005 across folds (pooled 0.991) if the column is an SD of replicates;
  0.965 ± 0.014 (pooled 0.972, worst fold 0.937) if it is already an SE.
- **Bulk, not tail:** the noisiest 5% hold about 21% of the total noise. Dropping them moves the
  SE-reading ceiling from 0.972 to 0.977.